# Validación del proyecto ECU 911

Comparo los dos mejores modelos de la sesión anterior con validación al azar y temporal. La temporal es la importante para este proyecto, porque el modelo se usaría para fechas futuras. Uso tres pliegues para cuidar los recursos de mi computadora sin quedarme con una sola división.

In [1]:
import time

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

from pipeline_proyecto import (
    CATEGORICAS, COLUMNAS_ARCHIVO, NUMERICAS, OBJETIVO, RUTA_DATOS,
    YA_LISTAS, agregar_features, construir_preparacion,
)

datos = pd.read_csv(
    RUTA_DATOS, usecols=COLUMNAS_ARCHIVO,
    dtype={"cod_canton": str}, parse_dates=["fecha"],
)
datos = agregar_features(datos).dropna(subset=[OBJETIVO])
datos = datos.sort_values(["fecha", "cod_canton"]).reset_index(drop=True)
X = datos[NUMERICAS + CATEGORICAS + YA_LISTAS]
y = datos[OBJETIVO]

corridas = pd.read_csv("corridas_proyecto.csv").sort_values("metrics.mae_prueba")
mejores = corridas["tags.mlflow.runName"].head(2).tolist()
print("Datos:", len(datos), "filas, desde", datos.fecha.min().date(), "hasta", datos.fecha.max().date())
print("Dos mejores según la sesión anterior:", mejores)

Datos: 40320 filas, desde 2026-04-01 hasta 2026-09-27
Dos mejores según la sesión anterior: ['red neuronal', 'bosque aleatorio']


## Métrica del negocio

Para planificación de emergencias considero que **subestimar es dos veces más caro** que sobreestimar. Si pronostico menos emergencias de las que llegan puede faltar personal o vehículos; si pronostico de más, el costo principal es tener capacidad ociosa.

Además del MAE calculo un costo asimétrico por fila: cada emergencia subestimada pesa 2 y cada una sobreestimada pesa 1. No son dólares reales, sino unidades de costo para hacer explícita la prioridad.

In [2]:
modelos = {
    "bosque aleatorio": RandomForestRegressor(
        n_estimators=100, max_depth=14, min_samples_leaf=2,
        random_state=42, n_jobs=1,
    ),
    "red neuronal": MLPRegressor(
        hidden_layer_sizes=(64,), max_iter=300,
        early_stopping=True, random_state=42,
    ),
}


def costo_asimetrico(real, prediccion):
    diferencia = np.asarray(real) - np.asarray(prediccion)
    return np.mean(np.where(diferencia > 0, 2 * diferencia, -diferencia))


def evaluar_cv(nombre_modelo, modelo, nombre_esquema, esquema):
    filas = []
    for numero, (idx_ent, idx_val) in enumerate(esquema.split(X, y), start=1):
        pipeline = Pipeline([
            ("preparar", construir_preparacion()),
            ("modelo", clone(modelo)),
        ])
        inicio = time.perf_counter()
        pipeline.fit(X.iloc[idx_ent], y.iloc[idx_ent])
        prediccion = pipeline.predict(X.iloc[idx_val])
        filas.append({
            "modelo": nombre_modelo,
            "validacion": nombre_esquema,
            "pliegue": numero,
            "mae": mean_absolute_error(y.iloc[idx_val], prediccion),
            "costo_asimetrico": costo_asimetrico(y.iloc[idx_val], prediccion),
            "filas_entrenamiento": len(idx_ent),
            "filas_validacion": len(idx_val),
            "segundos": time.perf_counter() - inicio,
        })
        print(nombre_modelo, nombre_esquema, "pliegue", numero, "listo")
    return filas

## Ejecución de los dos esquemas

La validación aleatoria mezcla fechas y sirve como comparación. `TimeSeriesSplit` no permite que el modelo vea el futuro al entrenar.

In [3]:
resultados = []
for nombre in mejores:
    resultados += evaluar_cv(
        nombre, modelos[nombre], "aleatoria",
        KFold(n_splits=3, shuffle=True, random_state=42),
    )
    resultados += evaluar_cv(
        nombre, modelos[nombre], "temporal",
        TimeSeriesSplit(n_splits=3),
    )

detalle = pd.DataFrame(resultados)
resumen = detalle.groupby(["modelo", "validacion"], as_index=False).agg(
    mae_promedio=("mae", "mean"),
    mae_desviacion=("mae", "std"),
    costo_promedio=("costo_asimetrico", "mean"),
    costo_desviacion=("costo_asimetrico", "std"),
)
pliegues = detalle.pivot_table(
    index=["modelo", "validacion"], columns="pliegue", values="mae"
).reset_index().rename(columns={1: "mae_pliegue_1", 2: "mae_pliegue_2", 3: "mae_pliegue_3"})
resumen = resumen.merge(pliegues, on=["modelo", "validacion"])
resumen.to_csv("resultados_validacion_proyecto.csv", index=False)

print()
print("Detalle por pliegue:")
print(detalle.round(4).to_string(index=False))
print()
print("Resumen:")
print(resumen.round(4).to_string(index=False))

red neuronal aleatoria pliegue 1 listo


red neuronal aleatoria pliegue 2 listo


red neuronal aleatoria pliegue 3 listo


red neuronal temporal pliegue 1 listo


red neuronal temporal pliegue 2 listo


red neuronal temporal pliegue 3 listo


... (2 outputs left out)

bosque aleatorio aleatoria pliegue 3 listo


bosque aleatorio temporal pliegue 1 listo


bosque aleatorio temporal pliegue 2 listo


bosque aleatorio temporal pliegue 3 listo

Detalle por pliegue:
          modelo validacion  pliegue    mae  costo_asimetrico  filas_entrenamiento  filas_validacion  segundos
    red neuronal  aleatoria        1 5.8327            8.8474                26880             13440   14.3186
    red neuronal  aleatoria        2 5.5228            8.2951                26880             13440   21.1581
    red neuronal  aleatoria        3 5.5841            8.1962                26880             13440   18.6985
    red neuronal   temporal        1 9.1765           14.6633                10080             10080    2.6304
    red neuronal   temporal        2 5.2935            8.1812                20160             10080   29.1394
    red neuronal   temporal        3 4.9402            7.5744                30240             10080   23.2668
bosque aleatorio  aleatoria        1 5.6121            8.5614                26880             13440   28.6086
bosque aleatorio  aleatoria        2 5.4437     

## Conclusión

El bosque quedó mejor en los dos esquemas. En la validación temporal obtuvo MAE 5.6736 y costo 8.3965, mientras la red obtuvo MAE 6.4700 y costo 10.1396. Por eso cambio la elección inicial y me quedo con el bosque: fue más estable y se equivocó menos en el error que considero más costoso. Los números completos quedan en `resultados_validacion_proyecto.csv`.